In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.optim import AdamW
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from sklearn.metrics import f1_score
from pytorch_lightning.callbacks import TQDMProgressBar

In [2]:
class CustomDataset(Dataset):
    def __init__(self, df, id_col, content_col, target_col):
        self.df = df
        self.id_col = id_col
        self.content_col = content_col
        self.target_col = target_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        return {
            "pmcid": row[self.id_col],
            "content": str(row[self.content_col]),
            "class": row[self.target_col],
        }


In [3]:
class CustomCollator:
    def __init__(self, tokenizer: AutoTokenizer, max_seq_len: int):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __call__(self, input_batch: list[dict]) -> dict:
        sentences = [instance["content"] for instance in input_batch]
        targets = [instance["class"] for instance in input_batch]

        tokenized_batch = self.tokenizer(
            sentences,
            padding=True,  # Dynamic padding to longest in batch
            max_length=self.max_seq_len,
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": tokenized_batch["input_ids"],
            "attention_mask": tokenized_batch["attention_mask"],
            "targets": torch.tensor(targets, dtype=torch.float),
        }


In [ ]:

# Model for classification


class BERTModel(pl.LightningModule):
    def __init__(self, model_name: str, num_classes: int, lr: float = 2e-5, class_weights=None):
        super().__init__()
        self.save_hyperparameters()
        self.model = AutoModel.from_pretrained(model_name)
        self.output_layer = nn.Linear(self.model.config.hidden_size, num_classes)
        self.register_buffer("class_weights", class_weights)
        self.loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        self.lr = lr

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.last_hidden_state[:, 0, :]
        return self.output_layer(cls_output)

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = self.loss_fct(logits, batch["targets"].long())
        preds = torch.argmax(logits, dim=1)

        acc = (preds == batch["targets"]).float().mean()
        f1 = f1_score(batch["targets"].cpu(), preds.cpu(), average="weighted")

        self.log("train_f1", f1, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        loss = self.loss_fct(logits, batch["targets"].long())
        preds = torch.argmax(logits, dim=1)

        acc = (preds == batch["targets"]).float().mean()
        f1 = f1_score(batch["targets"].cpu(), preds.cpu(), average="weighted")

        self.log("val_f1", f1, prog_bar=True, on_epoch=True)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=self.lr)


In [ ]:

# Assuming these are in your bert_base.py


def train(
    train_df,
    num_classes,
    num_epochs=5,
    id_col="pmcid",
    content_col="text",
    target_col="label",
):
    # 1. Setup Data
    print("Setting up data...")
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    collator = CustomCollator(tokenizer, max_seq_len=64)  # Reduced for speed

    train_loader = DataLoader(
        CustomDataset(train_df, id_col, content_col, target_col),
        batch_size=16,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,  # Must be 0 in notebooks (multiprocessing can't pickle notebook-defined classes)
        persistent_workers=False,
    )
    print(f"Train set size: {len(train_df)} samples, {len(train_loader)} batches")

    # 2. Setup Model
    print("Loading model...")
    weights = compute_class_weight("balanced", classes=np.unique(train_df[target_col]), y=train_df[target_col])
    class_weights = torch.tensor(weights, dtype=torch.float)
    model = BERTModel("bert-base-uncased", num_classes=num_classes, class_weights=class_weights)
    print(f"Model loaded: bert-base-uncased | num_classes={num_classes} | lr={model.lr}")

    # 3. Setup Trainer
    trainer = pl.Trainer(
        accelerator="auto",
        devices=1,
        max_epochs=num_epochs,
        num_nodes=1,
        logger=False,
        enable_checkpointing=False,
        enable_model_summary=False,
        enable_progress_bar=True,
        callbacks=[TQDMProgressBar(refresh_rate=1)],
    )

    # 4. Train
    print(f"Starting training for {num_epochs} epoch(s)...")
    trainer.fit(model, train_loader)
    print("Training complete.")
    return model, tokenizer


def predict(model, tokenizer, sentence, device):
    """Predicts a single class for a single sentence."""

    model.eval()

    inputs = tokenizer(
        sentence, return_tensors="pt", truncation=True, padding=True, max_length=64
    ).to(device)

    with torch.no_grad():
        logits = model(inputs["input_ids"], inputs["attention_mask"])

    return torch.argmax(logits, dim=1).item()


def evaluate(model, test_df, tokenizer):
    """Loops through a test dataframe and prints the final accuracy."""

    model.eval()

    true_labels = test_df["label"].tolist()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    predictions = []
    for _, row in test_df.iterrows():
        pred = predict(model, tokenizer, str(row["text"]), device)
        predictions.append(pred)

    f1 = f1_score(true_labels, predictions, average="weighted")
    print(f"F1 Score: {f1:.4f}")

    return f1


In [ ]:

def train_and_eval_on_train_set(train_data, save_dir="saved_model"):
    train_df, test_df = train_test_split(train_data, test_size=0.2, random_state=42)

    model, tokenizer = train(train_df, target_col="label", num_classes=2, num_epochs=3)

    evaluate(model, test_df, tokenizer)

    torch.save(model.state_dict(), f"{save_dir}/model.pt")
    tokenizer.save_pretrained(save_dir)
    print(f"Model and tokenizer saved to {save_dir}/")


In [7]:
if __name__ == "__main__":
    train_path = "data/train.tsv"
    val_path = "data/val.tsv"

    train_data = pd.read_csv(train_path, sep="\t")
    val_data = pd.read_csv(val_path, sep="\t")

    train_and_eval_on_train_set(train_data)

Setting up data...


Train set size: 12403 samples, 776 batches
Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Model loaded: bert-base-uncased | num_classes=2 | lr=2e-05
Starting training for 3 epoch(s)...


/Users/alinstefanescu/Documents/bionlp/.venv/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/alinstefanescu/Documents/bionlp/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/alinstefanescu/Documents/bionlp/.venv/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:534: Found 228 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/Users/alinstefanescu/Documents/bionlp/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
